In [42]:
from openai import OpenAI
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

In [29]:
from dotenv import load_dotenv
import os

load_dotenv()  # busca automáticamente el archivo .env en el directorio actual

api_key = os.getenv("OPENAI_API_KEY")

In [30]:
dataset = pd.read_csv("../demo_datasets/demo_tortugas.csv", sep=";").head()
print(dataset)

   id             ficha          especie nombre.especie  fecha_orig  \
0   1  1892 - 1977-1990  Caretta caretta   Tortuga Boba  18/04/1990   
1   2  1868 - 1977-1990  Caretta caretta   Tortuga Boba  14/11/1989   
2   3  9537 - 1998-2010  Caretta caretta   Tortuga Boba  02/12/2010   
3   4  9521 - 1998-2010  Caretta caretta   Tortuga Boba  15/11/2010   
4   5  9436 - 1998-2010  Caretta caretta   Tortuga Boba  29/09/2010   

        fecha  anio         mes   estacion               lugar_orig  ...  \
0  18/04/1990  1990       Abril  Primavera         Rambla de castro  ...   
1  14/11/1989  1989   Noviembre      Otoño                      NaN  ...   
2  02/12/2010  2010   Diciembre   Invierno  CANDELARIA - CANDELARIA  ...   
3  15/11/2010  2010   Noviembre      Otoño             Puerto Colón  ...   
4  29/09/2010  2010  Septiembre      Otoño         EL PORIS - ARICO  ...   

                                           fmt_lugar               muni  \
0  Calle Castro, 38611, Granadilla de Abo

In [31]:
# Función para leer el contenido de un archivo prompt:
def load_prompt(file_path):
    with open(file_path, 'r') as file:
        return file.read()

In [32]:
system_content = load_prompt("../prompts/llm_system_promt.txt")
system_content

'Eres un asistente inteligente encargado de extraer información de incidencias con fauna silvestre de una base de datos de un centro de recuperación de fauna silvestre.\n\nCada entrada es un texto que describe el incidente con tipo de animal, en concreto tortugas marinas. Proporciona una tabla estructurada que resuma la información sobre los incidentes, incluyendo columnas:\n\n- id: identificador de la incidencia con el código numérico inicial especificado al comienzo del texto de la incidencia.\n- partes_cuerpo: partes del cuerpo del animal que han sido afectadas, deben sestar etiquetadas con pocas palabras (por ejemplo, cabeza, ojos, aletas, ano, caparazón, cuerpo entero, etc..., sin especificar izquierda o derecha, delantera o trasera, superior o inferior, sólo de forma genérica) y en caso de no detectar la parte del cuerpo etiquetar como "indeterminado".\n- daño_enfermedad: los daños o enfermedades encontrados en cada una de dichas partes deben describirse con uno o dos términos se

In [33]:
def extract_data_from_text(text):
    # Load system content and user content
    #system_content = load_prompt('LLM/llm_system_prompt.txt')
    user_content = f"Text: {text}"

    # Initialize the OpenAI API client
    client = OpenAI()

    # Make a request to the OpenAI API to generate a chat completion
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": system_content
            },
            {
                "role": "user",
                "content": user_content
            }
        ],
        #model="gpt-3.5-turbo",
        model="gpt-4.1-mini",
        temperature=0.3,
        max_tokens=4096,
        top_p=0.8,
        response_format={ "type": "json_object" }
    )

    # Extract the completion result and token usage information from the response
    completion = chat_completion.choices[0].message.content
    result = json.loads(completion)
    prompt_tokens_used = chat_completion.usage.prompt_tokens
    completion_tokens_used = chat_completion.usage.completion_tokens

    return result, prompt_tokens_used, completion_tokens_used

In [34]:
## Ejemplo con cadenas de texto sueltas.

text = "Muy delgada baja movilidad, astenia y anorexia. Se diagnostica desnutricion"
text = "8. ¡¡ no coments, qué olor !!!!"
result, prompt_tokens_used, completion_tokens_used = extract_data_from_text(text)

print(f"Result : {result}")
print(f"Prompt Tokens Used : {prompt_tokens_used}")
print(f"Completion Tokens Used : {completion_tokens_used}")

Result : {'id': 8, 'partes_cuerpo': 'indeterminado', 'daño_enfermedad': 'indeterminado', 'causante': 'indeterminado', 'gravedad': 'indeterminado', 'estado': 'indeterminado', 'recogido': 'indeterminado', 'observa': 'Olor muy intenso'}
Prompt Tokens Used : 945
Completion Tokens Used : 77


In [36]:
df_sample = pd.DataFrame(['3. Le falta un trozo de caparazón posterior, mordida de tiburón, con rafia y mucho musgo, necrosada toda la parte posterior, separada la cloaca.',
'4. Anzuelo de palangre en el esófago, hemorragia abundante. Se opera para extraerle el anzuelo y nuere en una hora.',
'5. Con percebes, aparentemente bien, recogida en alta mar.',
#'6. Problema en ojo izdo, nariz y aleta delantera izda. Recogida por un pescador.',
#'7. Con nylon en las aletas, corte en la aleta delantera derecha y problemas en los ojos, muy bébil.',
'8. ¡¡ no coments, qué olor !!!!',
#'9. Se la encontraron flotando a la deriva'
'10. Anzuelo clavado. Recogida en Capitanía de Pto. Colón.  Peso:11,800 kgr.',
'11. Llena de algas',
'12. Musgo y percebes en caparazón. Trozo de caparazón mordido.'
],columns=['text'])
df_sample

,text
0,"3. Le falta un trozo de caparazón posterior, m..."
1,"4. Anzuelo de palangre en el esófago, hemorrag..."
2,"5. Con percebes, aparentemente bien, recogida ..."
3,"8. ¡¡ no coments, qué olor !!!!"
4,10. Anzuelo clavado. Recogida en Capitanía de ...
5,11. Llena de algas
6,12. Musgo y percebes en caparazón. Trozo de ca...


In [48]:
results = []
total_prompt_tokens_used = 0
total_completion_tokens_used = 0

for text in tqdm(df_sample['text'], desc="Processing texts"):
    result, prompt_tokens_used, completion_tokens_used = extract_data_from_text(text)
    results.append(result)

    total_prompt_tokens_used += prompt_tokens_used
    total_completion_tokens_used += completion_tokens_used

df_result = pd.DataFrame(results)

Processing texts: 100%|██████████| 7/7 [00:12<00:00,  1.82s/it]


In [ ]:
## Prueba 1
df_result

,id,partes_cuerpo,daño_enfermedad,causante,gravedad,estado,recogido,observa
0,3,caparazón,"herida, necrosis",tiburón,alta,indeterminado,indeterminado,"Falta un trozo de caparazón posterior, necrosa..."
1,4,esófago,"herida, hemorragia",anzuelo de palangre,fatal,muerto,indeterminado,"Operado para extraer el anzuelo, murió en una ..."
2,5,indeterminado,parasitacion,percebes,baja,indeterminado,indeterminado,recogida en alta mar
3,8,indeterminado,putrefacción,indeterminado,fatal,muerto,indeterminado,Olor muy intenso
4,10,indeterminado,anzuelo clavado,anzuelo,media,indeterminado,Capitanía de Pto. Colón,"Peso: 11,800 kgr."
5,11,indeterminado,contaminación,algas,baja,indeterminado,indeterminado,Llena de algas
6,12,caparazón,"parasitacion, mordida","percebes, musgo, indeterminado",media,indeterminado,indeterminado,Trozo de caparazón mordido


In [49]:
## Prueba 2
df_result

,id,partes_cuerpo,daño_enfermedad,causante,gravedad,estado,recogido,observa
0,3,caparazón,"falta, necrosis",tiburón,alta,indeterminado,indeterminado,"Presencia de rafia y mucho musgo, parte poster..."
1,4,esófago,hemorragia,anzuelo de palangre,fatal,muerto,indeterminado,Se operó para extraer el anzuelo pero murió en...
2,5,indeterminado,parasitacion,percebes,baja,indeterminado,indeterminado,recogida en alta mar
3,8,indeterminado,putrefacción,indeterminado,fatal,muerto,indeterminado,Olor muy intenso
4,10,indeterminado,anzuelo clavado,anzuelo,media,indeterminado,Capitanía de Pto. Colón,"Peso: 11,800 kgr."
5,11,indeterminado,contaminación,algas,baja,indeterminado,indeterminado,Llena de algas
6,12,caparazón,"parasitacion, mordida","percebes, musgo, indeterminado",media,indeterminado,indeterminado,Trozo de caparazón mordido


In [ ]:
## Probamos con la demo de los data sets
